# Seccion Global

In [ ]:
!pip install wordcloud
!pip install -U spacy
!python -m spacy download es

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import spacy
from wordcloud import WordCloud

In [ ]:
nlp = spacy.load('es_core_news_sm')

def lemmatize_spacy(text):
  doc = nlp(text)
  words_result = []
  for word in doc:
    words_result.append({"word": word.text,  "pos": word.pos_, "dep": word.dep_, "lemma": word.lemma_, "alpha": word.is_alpha})
  return words_result

lemmatize_spacy(u"El ejemplo dado en cada uno de los padres debe ser importante en los niños que estan corriendo.")

In [ ]:
df = pd.read_excel('../data/raw/proyecto_emociones/2024/DATOS ANALISIS/0 REPRESENTO LAS EMOCIONES.xlsx', sheet_name='Hoja2')
df

In [ ]:
df.Preg.unique()

In [ ]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk  import FreqDist, ConditionalFreqDist, bigrams
import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab') ## <- Esta fue la que se actulizo
stop_words = set(stopwords.words('spanish'))

In [ ]:
def clean_text(row):
  tokens_doc = word_tokenize(str(row['Res']).lower())
  tokens_doc_wo_stop = [w for w in tokens_doc if w.isalpha()]
  tokens_doc_wo_stop = [w for w in tokens_doc_wo_stop if not w.lower() in stop_words]
  return " ".join(tokens_doc_wo_stop)

def clean_text_lemma(row):
  tokens_doc = lemmatize_spacy(str(row['Res']))
  tokens_doc_wo_stop = [w['lemma'] for w in tokens_doc if w['alpha']]
  tokens_doc_wo_stop = [w for w in tokens_doc_wo_stop if not w.lower() in stop_words]
  return " ".join(tokens_doc_wo_stop)

def clean_text_lemma_bigram(row):
  tokens_doc = lemmatize_spacy(str(row['Res']))
  tokens_doc_wo_stop = [w['lemma'] for w in tokens_doc if w['alpha']]
  tokens_doc_wo_stop = [w for w in tokens_doc_wo_stop if not w.lower() in stop_words]
  return " ".join([w[0] +'_'+ w[1] for w in list(bigrams(tokens_doc_wo_stop))])

def clean_text_noun(row):
  tokens_doc = lemmatize_spacy(str(row['Res']))
  tokens_doc_wo_stop = [w['lemma'] for w in tokens_doc if w['pos'] == 'NOUN']
  tokens_doc_wo_stop = [w for w in tokens_doc_wo_stop if not w.lower() in stop_words]
  return " ".join(tokens_doc_wo_stop)

def clean_text_verb(row):
  tokens_doc = lemmatize_spacy(str(row['Res']))
  tokens_doc_wo_stop = [w['lemma'] for w in tokens_doc if w['pos'] == 'VERB']
  tokens_doc_wo_stop = [w for w in tokens_doc_wo_stop if not w.lower() in stop_words]
  return " ".join(tokens_doc_wo_stop)

def clean_text_adj(row):
  tokens_doc = lemmatize_spacy(str(row['Res']))
  tokens_doc_wo_stop = [w['lemma'] for w in tokens_doc if w['pos'] == 'ADJ']
  tokens_doc_wo_stop = [w for w in tokens_doc_wo_stop if not w.lower() in stop_words]
  return " ".join(tokens_doc_wo_stop)

df['txt_1'] = df.apply(clean_text, axis=1)
df['txt_2'] = df.apply(clean_text_lemma, axis=1)
df['txt_3'] = df.apply(clean_text_lemma_bigram, axis=1)
df['txt_verb'] = df.apply(clean_text_verb, axis=1)
df['txt_noun'] = df.apply(clean_text_noun, axis=1)
df['txt_adj'] = df.apply(clean_text_adj, axis=1)

In [ ]:
df

In [ ]:
import matplotlib.pyplot as plt
fdist1 = None
def freq_dist_top_100_monogram(text):
  fig = plt.figure(figsize = (20, 7))
  fdist1_local = FreqDist(word_tokenize(text))
  fdist1_local.plot(100)
  print("most_common(50):")
  #print(fdist1.most_common(50))
  fdist1_local.tabulate(50)
  return fdist1_local

fdist1 = freq_dist_top_100_monogram(" ".join(df['txt_2'].values).lower())

In [ ]:
#fdist1.most_common(50)

In [ ]:
def freq_dist_top_100_bigram(text):
  fig = plt.figure(figsize = (20, 7))
  fdist2 = FreqDist(bigrams(word_tokenize(text)))
  fdist2.plot(100)
  print("most_common(50):")
  print(fdist2.tabulate(50))

freq_dist_top_100_bigram(" ".join(df['txt_2'].values).lower())


In [ ]:
list(df['NUM'].unique())

In [ ]:
list(df['Preg'].unique())

In [ ]:
list_sample = [w for w, v in fdist1.most_common(50)]
print(fdist1.most_common(50))
len(list_sample)

In [ ]:
def cond_freq_dist(name_col):
  fig = plt.figure(figsize = (20, 8))
  cfd = nltk.ConditionalFreqDist(
      (fileid, word)
      for fileid in list(df['NUM'].unique())
      for word in word_tokenize(" ".join(df[df['NUM'] == fileid][name_col].values)))
  cfd.plot(samples=list_sample)
cond_freq_dist('txt_2')

In [ ]:
def cond_freq_dist(name_col):
  fig = plt.figure(figsize = (20, 8))
  cfd = nltk.ConditionalFreqDist(
      (fileid, word)
      for fileid in list(df['Preg'].unique())
      for word in word_tokenize(" ".join(df[df['Preg'] == fileid][name_col].values)))
  cfd.plot(samples=list_sample)
  print(cfd.tabulate(samples=list_sample))
cond_freq_dist('txt_2')

In [ ]:
def cloud_education(title, text):
  wordcloud_monograma = WordCloud(background_color ='white', width = 1000, height = 800).generate(text)
  fig = plt.figure(figsize = (20, 7))
  fig.suptitle(title, fontsize=16)
  plt.imshow(wordcloud_monograma, interpolation='bilinear')
  plt.axis("off")

In [ ]:
text = " ".join(df['txt_2'].values)
cloud_education('Grafico de Monogramas', text)

In [ ]:
text_bigram = " ".join(df['txt_3'].values)
cloud_education('Grafico de Bigramas', text_bigram)

# Seccion Partes

## Analisis de Verbos

---



In [ ]:
freq_dist_top_100_monogram(" ".join(df['txt_verb'].values))
freq_dist_top_100_bigram(" ".join(df['txt_verb'].values))
cond_freq_dist('txt_verb')
cloud_education('Grafico de Monogramas - Verbos', " ".join(df['txt_verb'].values))

## Adjetivos

In [ ]:
freq_dist_top_100_monogram(" ".join(df['txt_adj'].values))
freq_dist_top_100_bigram(" ".join(df['txt_adj'].values))
cond_freq_dist('txt_adj')
cloud_education('Grafico de Monogramas - Adjetivos', " ".join(df['txt_adj'].values))

## Sustativos

In [ ]:
freq_dist_top_100_monogram(" ".join(df['txt_noun'].values))
freq_dist_top_100_bigram(" ".join(df['txt_noun'].values))
cond_freq_dist('txt_noun')
cloud_education('Grafico de Monogramas - Sustantivos', " ".join(df['txt_noun'].values))

#Topicos

In [ ]:
!pip install bertopic
!pip install sentence-transformers

In [ ]:
def concatenate_column(df, column_name, separator=' '):
    concatenated_string = df[column_name].astype(str).str.cat(sep=separator)
    return concatenated_string


def group_preg(df):
    df_agrupado = df.groupby('Preg')['Res'].apply(lambda respuestas: ' '.join(respuestas)).reset_index()
    #df_agrupado.rename(columns={'Res': 'Resulta'}, inplace=True)
    return df_agrupado

In [ ]:
text_all =  concatenate_column(df, 'txt_2')
print(len(text_all))
text_all

In [ ]:
text_preg = group_preg(df)
text_preg

In [ ]:
from nltk.tokenize import word_tokenize

def dividir_en_bloques(texto, n_palabras=50):
    palabras = word_tokenize(texto)
    return [' '.join(palabras[i:i+n_palabras]) for i in range(0, len(palabras), n_palabras)]

docs = dividir_en_bloques(text_all, n_palabras=10)

In [ ]:
print(len(docs))
docs

In [ ]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from hdbscan import HDBSCAN

# Paso 1: Modelo de embeddings mejorado
embedding_model = SentenceTransformer('all-mpnet-base-v2')

# Paso 2: Configurar HDBSCAN PERMITIENDO prediction_data
hdbscan_model = HDBSCAN(min_cluster_size=2, min_samples=1, prediction_data=True)

# Paso 3: Crear BERTopic
topic_model = BERTopic(
    embedding_model=embedding_model,
    hdbscan_model=hdbscan_model,
    calculate_probabilities=True,
    verbose=True
)
topics, probs = topic_model.fit_transform(docs)

In [ ]:
# Ver resumen de temas
topic_model.get_topic_info()


In [ ]:
# Ver las palabras clave de un tema específico (ej. tema 0)
topic_model.get_topic(0)


In [ ]:
topic_model.visualize_topics()


In [ ]:
topic_model.visualize_barchart(top_n_topics=10)


# UMAP


In [ ]:
!pip install umap-learn[plot]


In [ ]:
import pandas as pd
import umap
import umap.plot
from sentence_transformers import SentenceTransformer
from bokeh.io import output_notebook, show

# Inicializar Bokeh en el notebook
output_notebook()

# 2. Generar embeddings
embedding_model = SentenceTransformer('all-mpnet-base-v2')
embeddings = embedding_model.encode(df["Res"].tolist())

# 3. Reducir dimensión con UMAP
umap_model = umap.UMAP(n_neighbors=5, n_components=2, metric='cosine')
embedding_2d = umap_model.fit_transform(embeddings)

# 4. Crear hover dataframe
hover_df = pd.DataFrame({
    'pregunta': df['Preg'],
    'respuesta': df['Res']
})


In [ ]:
# 5. Crear el gráfico interactivo
f = umap.plot.interactive(umap_model, labels=hover_df['pregunta'], hover_data=hover_df, point_size=5)

# 6. Mostrar el gráfico
show(f)